In [1]:
import json
from sklearn.metrics import cohen_kappa_score
# Round 1
annotated_ann1 = "data_40_ann1.json"
annotated_ann2 = "data_40_ann2.json"

with open(annotated_ann1, 'r', encoding='utf-8') as file:
    ann1 = json.load(file)


with open(annotated_ann2, 'r', encoding='utf-8') as file:
    ann2 = json.load(file)

In [2]:
categories = {
    "Misread Spec": "FN_misread_spec",
    "Added Requirement": "FN_added_requirement",
    "Overthink Edge": "FN_overthink_edge",
    "Assumed Type": "FN_assumed_type",
    "Imagined Runtime": "FN_imagined_runtime",
    "Performance Nitpick": "FN_performance_nitpick",
    "Readability Nitpick": "FN_readability_nitpick",
    "Precision Error": "FN_precision_error",
    "Boundary Error": "FN_boundary_error",
    "Logic Error": "FN_logic_error",
    "Vague Description": "FN_vague_description",
    "Other": "FN_other",
}
for t in categories.values():
    a = [ann1[i][t] for i in range(len(ann1))]
    w = [ann2[i][t] for i in range(len(ann2))]

    agree = sum(x == y for x, y in zip(a, w)) / len(ann1)
    k = cohen_kappa_score(a, w)
    print(f"{t:<15} \t kappa={k:6.3f}  agreement={agree:.2%} ann1_pos={sum(a):2d}  ann2_pos={sum(w):2d}")

FN_misread_spec 	 kappa= 0.521  agreement=82.50% ann1_pos=11  ann2_pos= 8
FN_added_requirement 	 kappa= 0.222  agreement=55.00% ann1_pos=11  ann2_pos=27
FN_overthink_edge 	 kappa= 0.143  agreement=77.50% ann1_pos=10  ann2_pos= 1
FN_assumed_type 	 kappa=   nan  agreement=100.00% ann1_pos= 0  ann2_pos= 0
FN_imagined_runtime 	 kappa=   nan  agreement=100.00% ann1_pos= 0  ann2_pos= 0
FN_performance_nitpick 	 kappa=   nan  agreement=100.00% ann1_pos= 0  ann2_pos= 0
FN_readability_nitpick 	 kappa=   nan  agreement=100.00% ann1_pos= 0  ann2_pos= 0
FN_precision_error 	 kappa=   nan  agreement=100.00% ann1_pos= 0  ann2_pos= 0
FN_boundary_error 	 kappa=   nan  agreement=100.00% ann1_pos= 0  ann2_pos= 0
FN_logic_error  	 kappa= 0.106  agreement=80.00% ann1_pos= 7  ann2_pos= 3
FN_vague_description 	 kappa= 1.000  agreement=100.00% ann1_pos= 1  ann2_pos= 1
FN_other        	 kappa=   nan  agreement=100.00% ann1_pos= 0  ann2_pos= 0


/opt/miniconda3/envs/dialogue/lib/python3.10/site-packages/sklearn/metrics/_classification.py:534: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/opt/miniconda3/envs/dialogue/lib/python3.10/site-packages/sklearn/metrics/_classification.py:897: RuntimeWarning: invalid value encountered in scalar divide
  k = np.sum(w_mat * confusion) / np.sum(w_mat * expected)
/opt/miniconda3/envs/dialogue/lib/python3.10/site-packages/sklearn/metrics/_classification.py:534: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/opt/miniconda3/envs/dialogue/lib/python3.10/site-packages/sklearn/metrics/_classification.py:897: RuntimeWarning: invalid value encountered in scalar divide
  k = np.sum(w_mat * confusion) / np.sum(w_mat * expect

In [3]:
ann1_by_id = {r["id"]: r for r in ann1}
ann2_by_id = {r["id"]: r for r in ann2}
ids = sorted(set(ann1_by_id) & set(ann2_by_id))

a = []
w = []
disagreements = {}
for i in ids:
    ann1_labels = [label for label, t in categories.items() if ann1_by_id[i][t]]
    ann2_labels = [label for label, t in categories.items() if ann2_by_id[i][t]]
    
    if len(ann1_labels) != 1:
        print('ann1 duplicate')
        print(f"id: {i}")
        print(ann1_labels)
        print(f"requirement: {ann1_by_id[i]['description']}")
        print()
    if len(ann2_labels) != 1: 
        print('ann2 duplicate')
        print(f"id: {i}")
        print(ann2_labels)
        print(f"requirement: {ann2_by_id[i]['description']}")
    
    ann1_labels = ann1_labels[0]
    ann2_labels = ann2_labels[0]
    if ann1_labels != ann2_labels:
        disagreements[i] = (ann1_labels, ann2_labels)
        
    a.append(ann1_labels)
    w.append(ann2_labels)


In [4]:
k = cohen_kappa_score(a, w)
k

0.2899408284023669

In [5]:
for i, (ann1_label, ann2_label) in disagreements.items():
    print(f"id: {i}")
    print(f"requirement: {ann1_by_id[i]['description']}")
    print(f"ann1: {ann1_label}")
    print(f"ann2: {ann2_label}")
    print()


id: 3
requirement: The system shall allow the use of PHI for treatment operations without requiring individual authorization.
ann1: Misread Spec
ann2: Added Requirement

id: 4
requirement: The system shall allow the disclosure of PHI for treatment operations without requiring individual authorization.
ann1: Misread Spec
ann2: Added Requirement

id: 10
requirement: The system shall support implementing policies and procedures to address security incidents.
ann1: Overthink Edge
ann2: Added Requirement

id: 11
requirement: The system shall identify and respond to suspected or known security incidents.
ann1: Logic Error
ann2: Added Requirement

id: 12
requirement: The system shall mitigate harmful effects to the extent practicable after a suspected or known security incidents.
ann1: Logic Error
ann2: Added Requirement

id: 17
requirement: The system shall implement procedures for safeguarding passwords.
ann1: Overthink Edge
ann2: Added Requirement

id: 18
requirement: The system shall supp

In [6]:
for i in ids:
    diffs = [t for t in categories.values() if bool(ann1_by_id[i][t]) != bool(ann2_by_id[i][t])]
    if not diffs:
        continue
    print(f"id: {i}")
    print(f"requirement: {ann1_by_id[i]['description']}")
    for t in diffs:
        who = "ann1" if ann1_by_id[i][t] else "ann2"
        print(f"  - {t}: True by {who}")
    for name, rec in (("ann1", ann1_by_id[i]), ("ann2", ann2_by_id[i])):
        note = (rec.get("notes") or rec.get("note") or "").strip()
        if note:
            print(f"  note ({name}): {note}")
    print()

id: 3
requirement: The system shall allow the use of PHI for treatment operations without requiring individual authorization.
  - FN_misread_spec: True by ann1
  - FN_added_requirement: True by ann2
  note (ann1): The requirement's scope is treatment operations, but the model reads it as governing PHI access generally, then rejects because non-treatment authenticated contexts can also reach PHI. It interprets a scoped permission (constraint A) as a global access rule (constraint B).

id: 4
requirement: The system shall allow the disclosure of PHI for treatment operations without requiring individual authorization.
  - FN_misread_spec: True by ann1
  - FN_added_requirement: True by ann2

id: 10
requirement: The system shall support implementing policies and procedures to address security incidents.
  - FN_added_requirement: True by ann2
  - FN_overthink_edge: True by ann1
  note (ann1): The model and GT say the same thing, but the model treats it as a generic and not sufficient requirem

# self consistency

In [7]:
categories = {
    "Consistent": "self_consistency_consistent",
    "Contradiction": "self_consistency_contradiction",
    "Unclear": "self_consistency_unclear",
}
for t in categories.values():
    a = [ann1[i][t] for i in range(len(ann1))]
    w = [ann2[i][t] for i in range(len(ann2))]

    agree = sum(x == y for x, y in zip(a, w)) / len(ann1)
    k = cohen_kappa_score(a, w)
    print(f"{t:<15} \t kappa={k:6.3f}  agreement={agree:.2%} ann1_pos={sum(a):2d}  ann2_pos={sum(w):2d}")

self_consistency_consistent 	 kappa= 0.375  agreement=92.50% ann1_pos=36  ann2_pos=39
self_consistency_contradiction 	 kappa=   nan  agreement=100.00% ann1_pos= 0  ann2_pos= 0
self_consistency_unclear 	 kappa=   nan  agreement=100.00% ann1_pos= 0  ann2_pos= 0


/opt/miniconda3/envs/dialogue/lib/python3.10/site-packages/sklearn/metrics/_classification.py:534: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/opt/miniconda3/envs/dialogue/lib/python3.10/site-packages/sklearn/metrics/_classification.py:897: RuntimeWarning: invalid value encountered in scalar divide
  k = np.sum(w_mat * confusion) / np.sum(w_mat * expected)
/opt/miniconda3/envs/dialogue/lib/python3.10/site-packages/sklearn/metrics/_classification.py:534: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/opt/miniconda3/envs/dialogue/lib/python3.10/site-packages/sklearn/metrics/_classification.py:897: RuntimeWarning: invalid value encountered in scalar divide
  k = np.sum(w_mat * confusion) / np.sum(w_mat * expect